# Phase 3 — Prompt Design & Evaluation

Testing the unified prompt that generates all 5 threat hunting output types.

**Model: claude-haiku-4-5-20251001** (dev mode — 20x cheaper for iteration)

This notebook:
1. Selects a stratified sample of 12-15 CAR analytics (4+ tactics, Windows + Linux)
2. Calls the unified prompt for each
3. Scores each output type (1-3 scale) against quality criteria
4. Computes mean scores and identifies failure patterns

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
from anthropic import Anthropic

from src.ingest import load_all_analytics
from src.prompts import PROMPT_UNIFIED, SYSTEM_PROMPT
from src.evaluate import OutputEvaluator, select_stratified_sample, compute_evaluation_summary

analytics = load_all_analytics()
client = Anthropic()
evaluator = OutputEvaluator()

print(f"Loaded {len(analytics)} analytics")

## 1. Select Stratified Sample

In [ ]:
sample = select_stratified_sample(analytics, n=12)

print(f"Selected {len(sample)} analytics for evaluation:")
print()

# Show coverage
tactics_covered = set()
platforms_covered = set()

for a in sample:
    print(f"{a['id']:20} {a['title'][:40]:40} Tactics: {','.join(a['tactics'][:2])}  Platforms: {','.join(a['platforms'])}")
    tactics_covered.update(a['tactics'])
    platforms_covered.update(a['platforms'])

print()
print(f"Tactics covered: {len(tactics_covered)} ({','.join(sorted(tactics_covered)[:4])}...)")
print(f"Platforms: {','.join(sorted(platforms_covered))}")

## 2. Generate Outputs for Each Analytic

Call the unified prompt for each analytic. This will take ~30-60 seconds.

Model: **claude-haiku-4-5-20251001** (dev mode)

In [ ]:
results = {}
scores = {}

for i, analytic in enumerate(sample, 1):
    print(f"[{i}/{len(sample)}] {analytic['id']} — {analytic['title'][:40]}...", end="", flush=True)

    # Build the unified prompt with analytic context
    analytic_context = f"""
CAR Analytic: {analytic['id']}
Title: {analytic['title']}
Description: {analytic['description']}
Platforms: {', '.join(analytic['platforms'])}
ATT&CK Techniques: {', '.join(analytic['techniques'])}
Tactics: {', '.join(analytic['tactics'])}
Data Model References: {', '.join(analytic['data_model_references']) if analytic['data_model_references'] else '(none)'}
Implementation Types: {', '.join(analytic['impl_types'])}
"""

    full_prompt = PROMPT_UNIFIED + "\n\n" + analytic_context

    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": full_prompt}]
        )

        output_text = response.content[0].text

        # Parse JSON from response
        try:
            output = json.loads(output_text)
        except json.JSONDecodeError:
            # If JSON parsing fails, try to extract JSON from the response
            start = output_text.find('{')
            end = output_text.rfind('}') + 1
            if start >= 0 and end > start:
                output = json.loads(output_text[start:end])
            else:
                output = {"error": "JSON parse failed"}
                print(" [PARSE ERROR]", flush=True)
                continue

        results[analytic['id']] = output
        scores[analytic['id']] = evaluator.evaluate_output(output, analytic)
        print(" [OK]", flush=True)

    except Exception as e:
        print(f" [ERROR: {str(e)[:30]}]", flush=True)
        continue

print(f"\nGenerated {len(results)} outputs successfully")

## 3. Score Outputs

In [ ]:
# Convert scores to DataFrame for easy viewing
score_df = pd.DataFrame(scores).T
score_df['mean'] = score_df.mean(axis=1)

print("Scores per analytic (1=poor, 2=adequate, 3=good):")
print(score_df.to_string())
print()

# Compute summary statistics
summary = compute_evaluation_summary(scores)

print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)
for output_type, stats in summary.items():
    print(f"\n{output_type}:")
    print(f"  Mean score: {stats['mean_score']} / 3.0")
    print(f"  Distribution: {stats['score_distribution']}")
    print(f"  Quality: ", end="")
    if stats['mean_score'] >= 2.5:
        print("GOOD")
    elif stats['mean_score'] >= 2.0:
        print("ADEQUATE")
    else:
        print("NEEDS WORK")

## 4. Identify Failure Patterns

In [ ]:
print("\n" + "="*70)
print("FAILURE PATTERN ANALYSIS")
print("="*70)

# Find output types that scored mostly 1s
for output_type, stats in summary.items():
    if stats['mean_score'] < 2.0:
        print(f"\n[PROBLEM] {output_type} averaging {stats['mean_score']:.2f}")
        ones = stats['score_distribution']['1']
        print(f"  {ones}/{stats['count']} outputs scored 1 (poor)")
        
        # Find which analytics had score=1
        bad_analytics = [aid for aid, s in scores.items() if s.get(output_type) == 1]
        if bad_analytics:
            print(f"  Failed on: {', '.join(bad_analytics[:3])}")
            
            # Show an example
            if bad_analytics[0] in results:
                example = results[bad_analytics[0]].get(output_type, {})
                if isinstance(example, dict):
                    print(f"  Example: {json.dumps(example, indent=2)[:200]}...")

print("\n" + "="*70)

## 5. Sample Outputs for Manual Review

Inspect a few complete outputs to spot patterns.

In [ ]:
# Show a high-scoring and a low-scoring example
if scores:
    # Find best and worst
    best_id = max(scores.keys(), key=lambda x: sum(scores[x].values()))
    worst_id = min(scores.keys(), key=lambda x: sum(scores[x].values()))
    
    print(f"BEST EXAMPLE: {best_id}")
    print(f"Score: {scores[best_id]}")
    print(json.dumps(results[best_id], indent=2)[:800])
    
    print("\n" + "="*70)
    print(f"WORST EXAMPLE: {worst_id}")
    print(f"Score: {scores[worst_id]}")
    print(json.dumps(results[worst_id], indent=2)[:800])